# பாடம் 03 - முகவர் வடிவமைப்பு செல்லுகள்

இந்த பாடத்தில், செயல்திறன் மிக்க AI முகவர்களை உருவாக்குவதற்கு மூன்று அடிப்படையான வடிவமைப்பு செல்லுக்களை பார்க்கிறோம்:

1. **தெளிவான முகவர் அறிவுரைகள்** — முகவர் நடத்தை வழிநடத்தும் துல்லியமான, வேலையை வரையறுக்கும் கூற்றுக்களை உருவாக்குதல்  
2. **Pydantic மாதிரிகளுடன் கட்டமைக்கப்பட்ட வெளியீடு** — முகவர்கள் முன்கூட்டியே எதிர்பாரக்கத்தக்க, சரிபார்க்கப்பட்ட தரவுகளைத் திருப்பிச் செய்வதை உறுதி செய்தல்  
3. **ஒரே பொறுப்புக்குரிய முகவர்கள்** — ஒவ்வொரு முகவரும் ஒரு காரியத்தை சிறப்பாக செய்ய வடிவமைத்தல்  

நாம் ஒவ்வொரு செல்லையும் **பயண இட பரிந்துரை வழங்குனர்** காட்சி நிலையில் பயன்படுத்தி, இடங்களை பரிந்துரைக்க, கிடைக்கும் நிலையை சரிபார்க்க மற்றும் லாஜிஸ்டிக்ஸை கையாளக்கூடிய ஒரு அமைப்பை படிப்படியாக கட்டமைப்போம்.


## அமைப்பு


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity pydantic --quiet

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated
from pydantic import BaseModel
from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

## Pattern 1: தெளிவான முகவர் அறிவுறுத்தல்கள்

அதிக பாதிப்பை ஏற்படுத்தும் உருவமைப்பு மிகவும் எளிமையானது: உங்கள் முகவருக்கு தெளிவான, விரிவான அறிவுறுத்தல்களை எழுதுவது.

நன்றாக அறிவுறுத்தல்கள் வரையறுக்கும்:
- **யார்** என்ற முகவர் (பात्रம் மற்றும் சுருதி)
- **என்ன** செய்ய வேண்டும் (படி படி பொறுப்புகள்)
- **எப்படி** செயல்பட வேண்டும் (கட்டுப்பாடுகள் மற்றும் பாணி)

கீழே, ஒவ்வொரு பதிலும் உருவாக்கும் தெளிவான அறிவுறுத்தல்களுடன் ஒரு பயண அனுசரணை முகவரியை உருவாக்குகிறோம்.


In [ ]:
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

agent = await provider.create_agent(
    name="TravelConcierge",
    instructions="""You are a luxury travel concierge named Alex. Your role is to:
1. Understand the traveler's preferences (budget, climate, activities)
2. Check destination availability before making recommendations
3. Provide detailed, personalized travel suggestions
4. Always mention visa requirements and best travel seasons
Be warm, professional, and enthusiastic about travel.""",
)

response = await agent.run(
    "I'd love a week-long vacation somewhere with great food and history. Budget around $2500."
)
print(response)

## பாட்டர்ன் 2: Pydantic மாதிரிகளுடன் கட்டமைக்கப்பட்ட வெளியீடு

இலவச வடிவ எழுத்து உரையாடலுக்கு பயனுள்ளதாக இருக்கிறது, ஆனால் கீழ் நிலை அமைப்புகளுக்கு கட்டமைக்கப்பட்ட தரவு தேவைப்படும்.  
**Pydantic மாதிரிகளை** ஒரு **கருவி செயல்பாட்டுடன்** இணைத்து, நாம் செய்ய முடியும்:

- முகவர் வெளியீட்டுக்கு சரியான திட்டத்தை வரையறுக்கவும்  
- பதில்களை தானாகச் சரிபார்க்கவும்  
- முகவர் முடிவுகளை பயன்பாட்டு நெறியில் நம்பகமாக ஒருங்கிணைக்கவும்  

நாம் ஒரு கருவியும் அறிமுகப்படுத்துகிறோம், இது முகவர் பரிந்துரைகளை உண்மையான தரவுகளில் உறுதிப்படுத்துவதற்கு இலக்கு விவரங்களை திருப்பி தருகிறது.


In [ ]:
class DestinationRecommendation(BaseModel):
    destination: str
    available: bool
    best_season: str
    highlights: list[str]
    estimated_budget_usd: int


class TravelRecommendations(BaseModel):
    recommendations: list[DestinationRecommendation]
    personalized_note: str


@tool(approval_mode="never_require")
def get_destination_details(destination: Annotated[str, "The destination to look up"]) -> str:
    """Get details about a vacation destination."""
    details = {
        "Barcelona": "Available. Best: May-Jun. Beach, architecture, nightlife. ~$2000/week",
        "Tokyo": "Available. Best: Mar-Apr. Culture, food, technology. ~$2500/week",
        "Cape Town": "Not available. Best: Nov-Mar. Nature, wine, adventure. ~$1800/week",
    }
    return details.get(destination, f"{destination}: No information available.")


structured_agent = await provider.create_agent(
    name="StructuredTravelExpert",
    instructions="You are a travel expert. Recommend destinations based on traveler preferences. Use the get_destination_details tool.",
    tools=[get_destination_details],
)

response = await structured_agent.run(
    "Recommend 3 destinations for a culture-loving traveler with a $2500 budget"
)

if response:
    print(response)

## Pattern 3: ஒரு பொறுப்புக் கொண்ட முகவர்கள்

சிக்கலான பணிகள் பல கவனம் செலுத்தும் முகவர்களுக்குள் பணி பிரிப்பதன் மூலம் பயனடைகின்றன, ஒவ்வொருவருக்கும் ஒரு பொறுப்பு உள்ளது:

- இடங்கள் மற்றும் கிடைக்கும் நிலையைப் பற்றி அறிவான **இடம் நிபுணர்**
- விமானங்கள், விடுதிகள் மற்றும் பயணத் திட்டங்களை கவனிக்கும் **வேலைத் திட்டமிடுபவர்**

இது மென்பொருள் உத்தியோக செயல்முறை *பிரிவினை தன்மை* என்ற கோட்பாட்டை பிரத பிரதிபலிக்கிறது — ஒவ்வொரு முகவரும் தனித்து சோதிக்க, பராமரிக்க மற்றும் மேம்படுத்த எளிதாக இருக்கிறது.


In [ ]:
destination_agent = await provider.create_agent(
    name="DestinationExpert",
    tools=[get_destination_details],
    instructions="""You are a destination research specialist. Your only job is to:
1. Evaluate destinations based on traveler preferences
2. Check availability using the provided tool
3. Return a short ranked list with pros/cons
Do NOT discuss flights, hotels, or logistics — another agent handles that.""",
)

logistics_agent = await provider.create_agent(
    name="LogisticsPlanner",
    instructions="""You are a travel logistics planner. Your only job is to:
1. Create a day-by-day itinerary for the chosen destination
2. Suggest flight and hotel options within the stated budget
3. Note visa requirements and travel insurance recommendations
Do NOT recommend destinations — another agent handles that.""",
)

# Step 1: Destination Expert picks the best options
dest_response = await destination_agent.run(
    "I want a week of culture and food for under $2500. Where should I go?"
)
print("=== Destination Expert ===")
print(dest_response)

# Step 2: Logistics Planner builds the trip plan
logistics_response = await logistics_agent.run(
    f"Plan a week-long trip based on this recommendation:\n{dest_response}"
)
print("\n=== Logistics Planner ===")
print(logistics_response)

## సారాంశం

ఈ పాఠంలో మేము ప్రయాణ సిఫార్సు సన్నివేశానికి మూడు ఏజంట్ డిజైన్ ప్యాటర్న్లను వర్తింపజేశాము:

| ప్యాటర్న్ | ప్రధాన ఆలోచన | లాభం |
|---|---|---|
| **స్పష్టమైన సూచనలు** | వ్యక్తిత్వం, బాధ్యతలు మరియు పరిమితులను ముందుగానే నిర్వచించండి | సुसంపన్నమైన, బ్రాండ్‌కు అనుగుణమైన ఏజెంట్ ప్రవర్తన |
| **సంరచనాత్మక అవుట్పుట్** | జవాబు ఫార్మాట్‌గా Pydantic నమూనాలను ఉపయోగించండి | ధ్రువీకరించబడిన, యంత్ర పఠనీయ ఫలితాలు |
| **ఒక్కటి బాధ్యత** | ప్రతి ఏజెంట్‌కు ఒక కేంద్రంగా పని ఇవ్వండి | పరీక్షించడం, నిర్వహించడం మరియు కలపడం సులభం |

ఈ ప్యాటర్న్లు సహజంగానే సంయుక్తమవుతాయి — మీరు స్పష్టమైన సూచనలను సంకలనం చేయవచ్చు, సంరచనాత్మక అవుట్పుట్‌ను ఒక ఒక్కడిపైకే బాధ్యత కలిగిన ఏజెంట్‌లో కలుపుకోవచ్చు, దీని ద్వారా బలమైన, ఉత్పత్తి-సిద్ధమైన వ్యవస్థలను నిర్మించడం సాధ్యం.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**மறுப்பு**:
இந்த ஆவணம் AI மொழிபெயர்ப்பு சேவை [Co-op Translator](https://github.com/Azure/co-op-translator) பயன்படுத்தி மொழிபெயர்க்கப்பட்டுள்ளது. நாங்கள் துல்லியத்தில்தான் முயலுகிறோம் என்றாலும், தானாக செய்யப்பட்ட மொழிபெயர்ப்புகளில் பிழைகள் அல்லது தவறுகள் இருக்க வாய்ப்பு உள்ளது என்பதை கவனத்தில் கொள்ளவும். சொந்த மொழியில் உள்ள அசலான ஆவணம் அதிகாரபூர்வ மூலமாகக் கருதப்பட வேண்டும். முக்கியமான தகவல்களுக்கு, தொழில்மறைய மனித மொழிபெயர்ப்பு பரிந்துரைக்கப்படுகிறது. இந்த மொழிபெயர்ப்பின் பயன்பாட்டினால் ஏற்படும் எந்தவொரு தவறான புரிதல் அல்லது தவறான விளக்கங்களுக்காகவும் நாங்கள் பொறுப்பாளர்கள் அல்லோம்.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
